# 🏥 Cirurgia Bariátrica no SUS — Análise de Mercado MedTech

**Workshop de Dados em Saúde | Inteligência de Mercado com Dados Públicos**

Este notebook extrai dados do SIH/DATASUS sobre cirurgias bariátricas pagas pelo SUS (2022–2024),  
gerando insumos para dashboards no Power BI sob a perspectiva de uma empresa MedTech.

---

### 🎯 Perguntas que queremos responder:
1. Quantas cirurgias bariátricas são realizadas pelo SUS por ano?
2. Quais estados concentram o maior volume?
3. Qual o perfil dos pacientes? (sexo, faixa etária, raça/cor)
4. Quais hospitais são líderes? (oportunidades de parceria comercial)
5. Quanto o SUS paga por procedimento?
6. Bypass ou Sleeve — qual é o mais realizado?

## ⚙️ 1. Instalação e Importações

In [ ]:
# Se estiver no Codespaces, as dependências já foram instaladas automaticamente.
# Caso contrário, rode:
# !pip install pysus pandas tqdm

import sys
sys.path.insert(0, '..')  # permite importar da pasta scripts/

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports OK')

: 

## 📥 2. Extração dos Dados

> **Atenção:** a extração completa (27 estados × 3 anos) pode levar 10–30 minutos  
> dependendo da sua conexão. Para testes, use `UFS_TESTE` abaixo.

In [ ]:
from pysus.ftp.databases.sih import SIH
from tqdm.notebook import tqdm
import os

# -------------------------------------------------------
# CONFIGURAÇÃO — ajuste conforme necessidade
# -------------------------------------------------------
CODIGOS_BARIATRICA = [
    '0407010090',  # Bypass Gástrico (Y-de-Roux)
    '0407010120',  # Gastrectomia Vertical (Sleeve)
    '0407010031',  # Bandagem Gástrica
    '0407010065',  # Derivação Biliopancreática
]

ANOS = [2022, 2023, 2024]

# Para testar rapidamente, use apenas SP e RJ:
UFS_TESTE = ['SP', 'RJ', 'MG']

# Para extração completa, descomente:
# UFS_TESTE = ['AC','AL','AP','AM','BA','CE','DF','ES','GO',
#              'MA','MT','MS','MG','PA','PB','PR','PE','PI',
#              'RJ','RN','RS','RO','RR','SC','SP','SE','TO']

REGIOES = {
    'AC':'Norte','AP':'Norte','AM':'Norte','PA':'Norte','RO':'Norte','RR':'Norte','TO':'Norte',
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MT':'Centro-Oeste','MS':'Centro-Oeste',
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','RS':'Sul','SC':'Sul',
}
PROCEDIMENTOS = {
    '0407010090': 'Bypass Gástrico (Y-de-Roux)',
    '0407010120': 'Gastrectomia Vertical (Sleeve)',
    '0407010031': 'Bandagem Gástrica',
    '0407010065': 'Derivação Biliopancreática',
}
SEXO    = {'1': 'Masculino', '3': 'Feminino'}
RACA_COR = {'01':'Branca','02':'Preta','03':'Parda','04':'Amarela','05':'Indígena','99':'Sem informação'}

print('Configuração OK — pronto para extrair!')

In [ ]:
def decode_idade(val):
    try:
        v = int(val)
        return v - 300 if v >= 300 else 0
    except:
        return None

def faixa_etaria(idade):
    if idade is None: return 'Sem informação'
    for a, b, label in [(0,17,'0-17'),(18,29,'18-29'),(30,39,'30-39'),
                        (40,49,'40-49'),(50,59,'50-59'),(60,200,'60+')]:
        if a <= idade <= b: return f'{label} anos'
    return 'Sem informação'

sih    = SIH().load()
frames = []
erros  = []

for ano in ANOS:
    print(f'\n📅 Ano {ano}')
    for uf in tqdm(UFS_TESTE, desc=f'UFs {ano}'):
        try:
            files  = sih.get_files('RD', uf=uf, year=ano)
            if not files: continue
            df_raw = sih.download(files).to_dataframe()

            col_proc = next((c for c in ['PROC_REA','PROC_SOLIC'] if c in df_raw.columns), None)
            if not col_proc: continue

            df_raw[col_proc] = df_raw[col_proc].astype(str).str.strip().str.zfill(10)
            df = df_raw[df_raw[col_proc].isin(CODIGOS_BARIATRICA)].copy()
            if df.empty: continue

            df = df.rename(columns={
                col_proc:'COD_PROCEDIMENTO', 'MUNIC_MOV':'COD_MUNICIPIO_HOSPITAL',
                'CNES':'COD_CNES_HOSPITAL',  'DIAS_PERM':'DIAS_INTERNACAO',
                'SEXO':'COD_SEXO',           'IDADE':'IDADE_RAW',
                'RACA_COR':'COD_RACA',       'VAL_TOT':'VALOR_TOTAL',
                'MORTE':'OBITO',             'DIAG_PRINC':'CID_PRINCIPAL',
            })

            df['UF']          = uf
            df['ANO']         = ano
            df['REGIAO']      = REGIOES.get(uf, '?')
            df['PROCEDIMENTO']= df['COD_PROCEDIMENTO'].map(PROCEDIMENTOS)

            if 'COD_SEXO'  in df.columns: df['SEXO']    = df['COD_SEXO'].astype(str).map(SEXO)
            if 'COD_RACA'  in df.columns: df['RACA_COR']= df['COD_RACA'].astype(str).str.zfill(2).map(RACA_COR)
            if 'IDADE_RAW' in df.columns:
                df['IDADE_ANOS']   = df['IDADE_RAW'].apply(decode_idade)
                df['FAIXA_ETARIA'] = df['IDADE_ANOS'].apply(faixa_etaria)
            if 'VALOR_TOTAL'      in df.columns: df['VALOR_TOTAL']      = pd.to_numeric(df['VALOR_TOTAL'], errors='coerce').fillna(0)
            if 'DIAS_INTERNACAO'  in df.columns: df['DIAS_INTERNACAO']  = pd.to_numeric(df['DIAS_INTERNACAO'], errors='coerce').fillna(0)
            if 'OBITO'            in df.columns: df['OBITO']            = df['OBITO'].astype(str).map({'1':'Sim','0':'Não'}).fillna('Não')

            frames.append(df)
            tqdm.write(f'  ✅ {uf}/{ano}: {len(df):,} registros')

        except Exception as e:
            erros.append(f'{uf}/{ano}: {e}')
            tqdm.write(f'  ⚠️ Erro {uf}/{ano}: {e}')

df_final = pd.concat(frames, ignore_index=True).sort_values(['ANO','UF']).reset_index(drop=True)
print(f'\n✅ Total extraído: {len(df_final):,} registros de cirurgia bariátrica')

## 🔍 3. Exploração Rápida dos Dados

In [ ]:
print('Shape:', df_final.shape)
df_final.head()

In [ ]:
# Volume por ano
df_final.groupby('ANO').size().rename('QTD_CIRURGIAS')

In [ ]:
# Top estados
df_final.groupby(['UF','REGIAO']).size().rename('QTD').sort_values(ascending=False).head(10)

In [ ]:
# Mix de procedimentos
df_final['PROCEDIMENTO'].value_counts(normalize=True).mul(100).round(1).rename('% do total')

In [ ]:
# Valor médio pago pelo SUS
df_final.groupby('PROCEDIMENTO')['VALOR_TOTAL'].agg(['mean','median','sum']).round(2)

## 💾 4. Exportar CSVs para o Power BI

In [ ]:
os.makedirs('../output', exist_ok=True)

def salvar(df, nome):
    path = f'../output/{nome}.csv'
    df.to_csv(path, index=False, encoding='utf-8-sig', sep=';')
    print(f'  💾 {nome}.csv ({len(df):,} linhas)')

# Base completa
salvar(df_final, 'bariatrica_sus_2022_2024')

# Resumo por UF/Ano
salvar(
    df_final.groupby(['ANO','REGIAO','UF'])
    .agg(QTD_CIRURGIAS=('UF','count'),
         VALOR_TOTAL_PAGO=('VALOR_TOTAL','sum'),
         VALOR_MEDIO=('VALOR_TOTAL','mean'),
         MEDIA_DIAS=('DIAS_INTERNACAO','mean'))
    .round(2).reset_index(),
    'resumo_por_uf_ano'
)

# Resumo por Procedimento
salvar(
    df_final.groupby(['ANO','PROCEDIMENTO'])
    .agg(QTD=('PROCEDIMENTO','count'),
         VALOR_TOTAL=('VALOR_TOTAL','sum'),
         VALOR_MEDIO=('VALOR_TOTAL','mean'))
    .round(2).reset_index(),
    'resumo_por_procedimento'
)

# Perfil demográfico
if {'SEXO','FAIXA_ETARIA','RACA_COR'}.issubset(df_final.columns):
    salvar(
        df_final.groupby(['ANO','SEXO','FAIXA_ETARIA','RACA_COR'])
        .agg(QTD=('UF','count')).reset_index(),
        'perfil_demografico'
    )

# Top hospitais
if 'COD_CNES_HOSPITAL' in df_final.columns:
    salvar(
        df_final.groupby(['COD_CNES_HOSPITAL','UF','REGIAO'])
        .agg(QTD_CIRURGIAS=('UF','count'),VALOR_TOTAL=('VALOR_TOTAL','sum'))
        .sort_values('QTD_CIRURGIAS', ascending=False).head(20).reset_index(),
        'top_hospitais'
    )

print('\n✅ Todos os arquivos salvos em /output/')

## 📊 5. Próximos Passos no Power BI

Importe os CSVs da pasta `/output/` usando **Obter Dados → Texto/CSV** com separador `;`.

| Arquivo | Visualização sugerida |
|---|---|
| `resumo_por_uf_ano.csv` | Mapa do Brasil preenchido por volume |
| `resumo_por_procedimento.csv` | Gráfico de rosca por tipo de cirurgia |
| `perfil_demografico.csv` | Pirâmide etária + segmentação por sexo |
| `top_hospitais.csv` | Tabela de oportunidades comerciais |
| `bariatrica_sus_2022_2024.csv` | Tabela detalhada com drill-through |

---

> 💡 **Dica MedTech:** O campo `COD_CNES_HOSPITAL` pode ser cruzado com o [CNES online](https://cnes.datasus.gov.br/) para obter nome, endereço e tipo de hospital — ideal para montar uma lista de targets comerciais.